# Fase 4 — Diagnostica Modello AE

> **Obiettivo**: capire **perché** l'autoencoder addestrato (`reports/checkpoints/ae_baseline.pth`) non separa normal da anomalo, usando 7 test (FASE 0 + A–G); ciascuno indaga un possibile punto di fallimento: non impara, overfitta, discriminazione per canale, drift, baseline grezza, capacità, sintesi.

> **Principi guida**
> - **Read-only**: niente scrittura su `src/`, `config/`, `data/processed/` o checkpoint; gli errori si ricalcolano in forward pass.
> - **Output isolati**: solo artefatti in `reports/diagnostics/model_diagnosis/` (CSV, PNG, JSON).
> - **Seed fissato**: `set_seed(42)` da `src/utils/seed.py`, dichiarato in ogni cella.
> - **Prevalenza** sempre mostrata accanto alle AUC (livello caso per la PR-AUC).
> - **Test set**: usato solo per la conferma finale (appendice G), mai per calibrare soglie.
> - **Conclusione**: ogni sezione chiude con *Cosa osservo / Cosa concludo / Cosa NON posso concludere*.

> **Forma di ogni sezione**: breve intro (cosa consiste / perché / come interpretare) → cella di codice opzionale → conclusione metodologica. Nessuna conclusione riporta i risultati attuali: il testo è generale così da valere per qualsiasi esecuzione.

In [ ]:
"""FASE 0 — Setup globale: seed, device, config, paths.

Esegui questa cella PRIMA di tutte le altre.
"""
import os
import sys
from pathlib import Path

import numpy as np
import torch

# Path resolution — works from project root or notebooks/
CWD = os.getcwd()
if "notebooks" in CWD:
    PROJECT_ROOT = os.path.dirname(CWD)
else:
    PROJECT_ROOT = CWD
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from src.utils.seed import set_seed
from src.utils.config import load_config, get_param

# --- Global constants ---
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Load config ---
config = load_config()  # carica config.yaml + params_validated_ae.yaml (merge)

# --- Paths ---
processed_dir = Path(PROJECT_ROOT) / "data" / "processed"
DIAG_DIR = Path(PROJECT_ROOT) / "reports" / "diagnostics" / "model_diagnosis"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

# --- Checkpoint file ---
CHECKPOINT_PATH = Path(PROJECT_ROOT) / "reports" / "checkpoints" / "ae_baseline.pth"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DEVICE: {DEVICE}")
print(f"SEED: {SEED}")
print(f"Config loaded: {list(config.keys())}")
print(f"processed_dir: {processed_dir}")
print(f"DIAG_DIR: {DIAG_DIR}")
print(f"CHECKPOINT_PATH: {CHECKPOINT_PATH}")
print(f"Checkpoint exists: {CHECKPOINT_PATH.exists()}")

In [ ]:
"""FASE 0 — Inventario artefatti nel checkpoint.

Carica reports/checkpoints/ae_baseline.pth (read-only) e produce una tabella
che dice cosa è presente/assente e quale sezione è eseguibile.

Non salva nulla sul checkpoint né modifica nessun file sorgente.
"""
set_seed(SEED)

# --- Load checkpoint (read-only) ---
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)

# --- Inspect keys ---
print("Chiavi nel checkpoint:")
for k in ckpt:
    v = ckpt[k]
    if k == "model_state_dict":
        n_params = sum(p.numel() for p in v.values())
        print(f"  {k}: {len(v)} parametri, {n_params} elementi totali")
    elif k == "history":
        train_l = v.get("train_loss", [])
        val_l = v.get("val_loss", [])
        be = v.get("best_epoch", "N/A")
        print(f"  history: train_loss={len(train_l)} voci, val_loss={len(val_l)} voci, best_epoch={be}")
    else:
        print(f"  {k}: {v}")

# --- Artifact status table ---
artifacts = {
    "model_state_dict": ckpt.get("model_state_dict") is not None,
    "input_dim": "input_dim" in ckpt,
    "window_size": "window_size" in ckpt,
    "latent_dim": "latent_dim" in ckpt,
    "conv_channels": "conv_channels" in ckpt,  # ATTESI: assente — usiamo default (64,32,16)
    "history (train_loss)": len(ckpt.get("history", {}).get("train_loss", [])) > 0,
    "history (val_loss)": len(ckpt.get("history", {}).get("val_loss", [])) > 0,
    "best_epoch": "best_epoch" in ckpt.get("history", {}),
    "seed": "seed" in ckpt,  # ATTESI: assente
    "config_yaml": "config" in ckpt or "config_yaml" in ckpt,  # ATTESI: assente
}

# --- Section executability ---
history_present = (
    artifacts["history (train_loss)"] and
    artifacts["history (val_loss)"] and
    artifacts["best_epoch"] and
    len(ckpt.get("history", {}).get("val_loss", [])) > 0
)

ARTIFACT_STATUS = {
    "A": True,  # MAE_zero vs MAE_AE — sempre eseguibile con modello
    "B": history_present,  # curve training — solo se history presente
    "C": True,  # errore per canale — sempre eseguibile
    "D": True,  # distribuzioni/tempo — sempre eseguibile
    "E": True,  # baseline senza rete — sempre eseguibile
    "F": True,  # overfit test — dentro notebook
    "G": True,  # sintesi — sempre eseguibile
}

print("\n" + "=" * 60)
print("ARTIFACT STATUS TABLE")
print("=" * 60)
print(f"{'Artefatto':<30} {'Presente':<12} {'Conseguenza':<30}")
print("-" * 60)
for art, present in artifacts.items():
    status = "✅ SÌ" if present else "❌ NO"
    if present:
        conseq = "OK"
    elif art == "conv_channels":
        conseq = "Usa default (64,32,16)"
    elif art == "seed":
        conseq = "FASE 0 → segnala"
    elif art in ("config_yaml",):
        conseq = "FASE 0 → segnala"
    elif "history" in art or "best_epoch" in art:
        conseq = f"Sezione B -> {'esegui' if history_present else 'in SOSPESO'}"
    else:
        conseq = "in SOSPESO"
    print(f"{art:<30} {status:<12} {conseq:<30}")

print("\n" + "=" * 60)
print("SEZIONI ESEGUIBILI")
print("=" * 60)
for sec in ["A", "B", "C", "D", "E", "F", "G"]:
    status = "✅ ESEGUI" if ARTIFACT_STATUS[sec] else "⏸ IN SOSPESO"
    print(f"  Sezione {sec}: {status}")

# Stampa valori chiave del checkpoint
W_ckpt = ckpt.get("window_size", "UNKNOWN")
input_dim_ckpt = ckpt.get("input_dim", "UNKNOWN")
latent_dim_ckpt = ckpt.get("latent_dim", "UNKNOWN")
print(f"\nDati checkpoint: W={W_ckpt}, input_dim={input_dim_ckpt}, latent_dim={latent_dim_ckpt}")

# Verifica coerenza con config
W_config = int(get_param(config, "model.window_size", 16))
print(f"Config params_validated_ae.yaml: window_size={W_config}")
if W_ckpt != W_config and W_ckpt != "UNKNOWN":
    print(f"⚠️  MISMATCH: checkpoint W={W_ckpt} ≠ config W={W_config}")

### Cosa osservo / Cosa concludo / Cosa NON posso concludere — FASE 0

**Cosa osservo**: il checkpoint contiene `model_state_dict`, `input_dim`, `window_size`, `latent_dim` e, se presente, `history` (train/val loss). Manca `seed` e la configurazione YAML completa.

**Cosa concludo**: le sezioni che usano solo i pesi (A, C, D, E, F, G) sono eseguibili — gli errori si ricalcolano con un forward pass. La sezione B (curve di training) è eseguibile **solo se `history` è presente e non vuoto**. La sezione F (overfit test) è sempre eseguibile inside-notebook, in un'eccezione all'obbligo read-only.

**Cosa NON posso concludere**: non so **quale seed** ha prodotto il checkpoint. Non so se `window_size` nel checkpoint è coerente con `params_validated_ae.yaml`. Non posso generalizzare ad altri seed né a configurazioni non registrate nel checkpoint stesso.

In [ ]:
"""Istantzia il modello caricando i pesi dal checkpoint + build loader.

Usa W, input_dim, latent_dim LETTI DAL CHECKPOINT (non da params.yaml).
conv_channels non è nel .pth → default (64, 32, 16) come in autoencoder.py.
"""
set_seed(SEED)

# --- Read model config from checkpoint (not from params.yaml) ---
W = ckpt["window_size"]
input_dim = ckpt.get("input_dim", 82)
latent_dim = ckpt.get("latent_dim", 16)

print(f"Modello istantiation: W={W}, input_dim={input_dim}, latent_dim={latent_dim}, conv_channels=(64,32,16)")

# --- Instantiate model with checkpoint params ---
# NOTA: SequenceAutoencoder usa conv_channels=(64,32,16) come default in autoencoder.py
from src.models.autoencoder import SequenceAutoencoder
model = SequenceAutoencoder(input_dim=input_dim, window_size=W, latent_dim=latent_dim)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE)
model.eval()
print(f"Modello caricato: {sum(p.numel() for p in model.parameters())} parametri totali")

# --- Load processed data ---
train_data = np.load(processed_dir / "train.npy")
val_normal_data = np.load(processed_dir / "val_normal.npy")
val_anomaly_data = np.load(processed_dir / "val_anomaly.npy")
test_normal_data = np.load(processed_dir / "test_normal.npy")
test_anomaly_data = np.load(processed_dir / "test_anomaly.npy")
selected_columns = np.load(processed_dir / "selected_columns.npy", allow_pickle=True)

print(f"\nData shapes: train={train_data.shape}, val_normal={val_normal_data.shape}, "
      f"val_anomaly={val_anomaly_data.shape}, test_normal={test_normal_data.shape}, "
      f"test_anomaly={test_anomaly_data.shape}")
print(f"Selected columns: {selected_columns.shape[0]} features")

# --- Build single-sample loaders (shuffle=False per regole leakage) ---
# NOTA: batch_size grande per velocità forward
from src.data.dataset import build_loader
BATCH_SIZE = 512

train_loader = build_loader(train_data, window_size=W, batch_size=BATCH_SIZE, label=0, shuffle=False, num_workers=0)
val_normal_loader = build_loader(val_normal_data, window_size=W, batch_size=BATCH_SIZE, label=0, num_workers=0)
val_anomaly_loader = build_loader(val_anomaly_data, window_size=W, batch_size=BATCH_SIZE, label=1, num_workers=0)
test_normal_loader = build_loader(test_normal_data, window_size=W, batch_size=BATCH_SIZE, label=0, num_workers=0)
test_anomaly_loader = build_loader(test_anomaly_data, window_size=W, batch_size=BATCH_SIZE, label=1, num_workers=0)

print(f"\nNumero finestre:")
print(f"  train:         {len(train_loader.dataset)}")
print(f"  val_normal:    {len(val_normal_loader.dataset)}")
print(f"  val_anomaly:   {len(val_anomaly_loader.dataset)}")
print(f"  test_normal:   {len(test_normal_loader.dataset)}")
print(f"  test_anomaly:  {len(test_anomaly_loader.dataset)}")

# Validation prevalence
n_val_normal = len(val_normal_loader.dataset)
n_val_anomaly = len(val_anomaly_loader.dataset)
val_prevalence = n_val_anomaly / (n_val_normal + n_val_anomaly)
print(f"\nPrevalenza val: {val_prevalence:.4f} ({n_val_anomaly}/{n_val_normal+n_val_anomaly})")

# Test prevalence
n_test_normal = len(test_normal_loader.dataset)
n_test_anomaly = len(test_anomaly_loader.dataset)
test_prevalence = n_test_anomaly / (n_test_normal + n_test_anomaly)
print(f"Prevalenza test: {test_prevalence:.4f} ({n_test_anomaly}/{n_test_normal+n_test_anomaly})")

print("\n✅ Modello + loader pronti. Usa DEVICE e compute_errors_on_loader per tutte le sezioni successive.")

## A) Baseline modello stupido — MAE_zero vs MAE_AE

**Cosa consiste**: si confronta il MAE reale della rete (MAE_AE) con il MAE ottenuto predendo sempre la media per-canale del training set (MAE_zero). Si calcola l'AUC del punteggio MAE, sia diretta che con segno invertito.

**Perché**: se `MAE_AE ≈ MAE_zero` la rete non ha imparato nulla (underfitting). Se invece `MAE_AE ≪ MAE_zero` ha imparato, e l'AUC (eventualmente invertita) dice se l'errore è discriminante.

**Come interpretare** (rapporto MAE_AE/MAE_zero con AUC diretta e AUC con segno invertito):
| Rapporto | AUC diretta | AUC invertita | Lettura |
|---|---|---|---|
| ≥ 0.8 | ≈ 0.5 | ≈ 0.5 | Non impara (underfitting) |
| ≪ 0.3 | < 0.45 | > 0.65 | Impara ma segno invertito |
| ≪ 0.3 | > 0.55 | — | Impara e discriminativo |

> MAE_zero è calcolato esplicitamente dai dati (media per-canale del train), non assunto nullo.

In [ ]:
"""Section A — Baseline "modello stupido": MAE_zero vs MAE_AE + AUC.

MAE_zero: MAE predendo la media per-canale del train (≈0 su dati standardizzati,
ma calcolata esplicitamente dai dati).

Salva risultati in reports/diagnostics/model_diagnosis/ae_section_a_results.csv
"""
set_seed(SEED)

# --- Import needed functions ---
from src.models.autoencoder import compute_errors_on_loader
from src.utils.metrics import calculate_auc
import csv

# --- Compute per-channel mean from train (media, non zero) ---
train_flat = train_data  # shape (n_samples, n_features)
channel_means = train_flat.mean(axis=0)  # shape (82,)
print(f"Media per-canale del train: shape={channel_means.shape}")
print(f"Max |media|: {np.abs(channel_means).max():.6e} (dovrebbe ≈ 0 su dati standardizzati)")

# --- MAE_ZERO: mean per-channel error if we predict channel_means ---
# Per ogni finestra del loader, la previsione è channel_means broadcastata
# MAE_zero = mean(|window - channel_means|) per finestra → skor per finestra

def compute_mae_zero(loader, channel_means, device=DEVICE):
    """Calcola MAE_zero per finestra: predice sempre channel_means (media train)."""
    errors = []
    for batch in loader:
        x = batch[0] if isinstance(batch, (list, tuple)) else batch
        # x: (B, W, input_dim) → predict channel_means broadcasted → (B, W, input_dim)
        x_pred = np.broadcast_to(channel_means, x.shape)
        x_np = x.numpy()
        mae_per_window = np.abs(x_np - x_pred).mean(axis=(1, 2))  # mean over W and features
        errors.append(mae_per_window)
    return np.concatenate(errors)

print("\nCalcolo MAE_zero...")
mae_zero_train = compute_mae_zero(train_loader, channel_means)
mae_zero_val_normal = compute_mae_zero(val_normal_loader, channel_means)
mae_zero_val_anomaly = compute_mae_zero(val_anomaly_loader, channel_means)

print(f"  MAE_zero train:        {mae_zero_train.mean():.6f}")
print(f"  MAE_zero val_normal:   {mae_zero_val_normal.mean():.6f}")
print(f"  MAE_zero val_anomaly:  {mae_zero_val_anomaly.mean():.6f}")

# --- MAE_AE: reconstruction error from the model ---
print("\nCalcolo MAE_AE (forward del modello)...")
mae_ae_train = compute_errors_on_loader(model, train_loader, DEVICE, metric="mae")
mae_ae_val_normal = compute_errors_on_loader(model, val_normal_loader, DEVICE, metric="mae")
mae_ae_val_anomaly = compute_errors_on_loader(model, val_anomaly_loader, DEVICE, metric="mae")

print(f"  MAE_AE train:        {mae_ae_train.mean():.6f}")
print(f"  MAE_AE val_normal:   {mae_ae_val_normal.mean():.6f}")
print(f"  MAE_AE val_anomaly:  {mae_ae_val_anomaly.mean():.6f}")

# --- Ratio MAE_AE / MAE_zero ---
ratio_train = mae_ae_train.mean() / mae_zero_train.mean()
ratio_val_norm = mae_ae_val_normal.mean() / mae_zero_val_normal.mean()
ratio_val_anom = mae_ae_val_anomaly.mean() / mae_zero_val_anomaly.mean()

print(f"\n--- Rapporto MAE_AE / MAE_zero ---")
print(f"  train:        {ratio_train:.4f}")
print(f"  val_normal:   {ratio_val_norm:.4f}")
print(f"  val_anomaly:  {ratio_val_anom:.4f}")
print(f"\nInterpretazione: ratio > 0.8 → rete impara quasi nulla; ratio < 0.3 → apprendimento significativo")

# --- AUC: MAE_zero as anomaly score (on validation) ---
val_errors_zero = np.concatenate([mae_zero_val_normal, mae_zero_val_anomaly])
val_errors_ae = np.concatenate([mae_ae_val_normal, mae_ae_val_anomaly])
val_labels = np.concatenate([
    np.zeros(len(mae_zero_val_normal), dtype=np.int32),
    np.ones(len(mae_zero_val_anomaly), dtype=np.int32),
])

# Per prevalenza
n0_val = len(mae_zero_val_normal)
n1_val = len(mae_zero_val_anomaly)
prevalence_val = n1_val / (n0_val + n1_val)
print(f"\nPrevalenza val: {prevalence_val:.4f} ({n1_val}/{n0_val+n1_val})")

auc_zero = calculate_auc(val_labels, val_errors_zero)
auc_ae = calculate_auc(val_labels, val_errors_ae)
auc_ae_inv = calculate_auc(val_labels, -val_errors_ae)  # segno invertito

print(f"\n--- AUC (validation: val_normal + val_anomaly) ---")
print(f"  MAE_zero:  ROC-AUC={auc_zero['roc_auc']:.4f}, PR-AUC={auc_zero['pr_auc']:.4f}  (prevalenza: {prevalence_val:.4f})")
print(f"  MAE_AE:    ROC-AUC={auc_ae['roc_auc']:.4f}, PR-AUC={auc_ae['pr_auc']:.4f}  (prevalenza: {prevalence_val:.4f})")
print(f"  MAE_AE(-): ROC-AUC={auc_ae_inv['roc_auc']:.4f}, PR-AUC={auc_ae_inv['pr_auc']:.4f}  (segno invertito)")
print(f"\nConfronto: AUC_AE / AUC_zero (ROC) = {auc_ae['roc_auc']/auc_zero['roc_auc']:.4f}")
print(f"Confronto: AUC_AE(-) / AUC_zero (ROC) = {auc_ae_inv['roc_auc']/auc_zero['roc_auc']:.4f}")

# --- Classification diagnostica ---
print(f"\n--- Diagnostica combinata ---")
if ratio_val_norm < 0.5 and ratio_val_anom < 0.5:
    print("  MAE_AE << MAE_zero → rete HA imparato a ricostruire (underfitting non è il problema)")
else:
    print("  MAE_AE ≈ MAE_zero → rete NON ha imparato (sospetto underfitting/corda)")

if auc_ae['roc_auc'] < 0.45:
    if auc_ae_inv['roc_auc'] > 0.65:
        print("  AUC AE invertita > 0.65 → rete impara segno INVERTO (bug di segno/convenzione)")
    else:
        print("  AUC AE bassa anche invertita → ampiezza errore non discriminativa")
elif auc_ae['roc_auc'] > 0.55:
    print("  AUC AE ≈ o > 0.55 → rete discriminativa (problema altrove: soglia, drift, ...)")

# --- Save results CSV ---
csv_path = DIAG_DIR / "ae_section_a_results.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["split", "MAE_zero", "MAE_AE", "ratio_AE_zero"])
    writer.writerow(["train", f"{mae_zero_train.mean():.6f}", f"{mae_ae_train.mean():.6f}", f"{ratio_train:.4f}"])
    writer.writerow(["val_normal", f"{mae_zero_val_normal.mean():.6f}", f"{mae_ae_val_normal.mean():.6f}", f"{ratio_val_norm:.4f}"])
    writer.writerow(["val_anomaly", f"{mae_zero_val_anomaly.mean():.6f}", f"{mae_ae_val_anomaly.mean():.6f}", f"{ratio_val_anom:.4f}"])
    writer.writerow([])
    writer.writerow(["score_type", "roc_auc", "pr_auc", "prevalence"])
    writer.writerow(["MAE_zero", f"{auc_zero['roc_auc']:.4f}", f"{auc_zero['pr_auc']:.4f}", f"{prevalence_val:.4f}"])
    writer.writerow(["MAE_AE", f"{auc_ae['roc_auc']:.4f}", f"{auc_ae['pr_auc']:.4f}", f"{prevalence_val:.4f}"])
    writer.writerow(["MAE_AE inverted", f"{auc_ae_inv['roc_auc']:.4f}", f"{auc_ae_inv['pr_auc']:.4f}", f"{prevalence_val:.4f}"])

print(f"\n📊 Risultati salvati in: {csv_path}")

# --- Plot MAE comparison ---
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
splits = ["train", "val_normal", "val_anomaly"]
mae_z = [mae_zero_train.mean(), mae_zero_val_normal.mean(), mae_zero_val_anomaly.mean()]
mae_a = [mae_ae_train.mean(), mae_ae_val_normal.mean(), mae_ae_val_anomaly.mean()]
ratios = [mae_a[i]/mae_z[i] for i in range(3)]

x = np.arange(len(splits))
w = 0.25
ax.bar(x - w, mae_z, w, label="MAE_zero (predice media train)", color="#aaa")
ax.bar(x, mae_a, w, label="MAE_AE (ricostruzione reale)", color="#2196F3")
ax.set_xlabel("Split")
ax.set_ylabel("MAE medio per finestra")
ax.set_title(f"MAE_zero vs MAE_AE (W={W}, input_dim={input_dim})\n"
              f"Rapporto MAE_AE/MAE_zero: train={ratios[0]:.3f}, val_norm={ratios[1]:.3f}, val_anom={ratios[2]:.3f}")
ax.set_xticks(x)
ax.set_xticklabels(splits)
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.savefig(DIAG_DIR / "ae_section_a_mae_comparison.png", dpi=120)
plt.show()

print("✅ Fine Sezione A")

### Cosa osservo / Cosa concludo / Cosa NON posso concludere — Sezione A

**Cosa osservo**: il rapporto `MAE_AE / MAE_zero` (valori bassi significano che la rete ricostruisce bene rispetto alla media) e l'AUC diretta e AUC con segno invertito, che mostrano se l'ampiezza dell'errore separa o meno le classi. La prevalenza del validation set è mostrata accanto alla PR-AUC.

**Cosa concludo**: un rapporto basso (≪ 0.3) indica che la rete ha imparato a ricostruire (nessun underfitting). Un AUC diretta bassa con AUC invertita alta indica una possibile inversione di segno; se invece l'AUC è bassa anche invertita, l'ampiezza non è discriminativa e serve cercare informazione in altra sezione (C per canali, E per baseline). Un AUC alto (≳ 0.55) indica che l'AE è discriminativo e il problema è altrove (soglia, drift, C/D).

**Cosa NON posso concludere**: questa risposta vale per un solo seed. Non attribuisco la debolezza a una causa unica senza le sezioni successive (C/D/E). Non so se un'architettura diversa cambi il rapporto (serve F).

## B) Curve di training — Richiede `history` nel checkpoint

**Cosa consiste**: si osservano le curve `train_loss` e `val_loss` per epoca, con `best_epoch` evidenziato.

**Perché**: la pendenza e il divario tra le due curve rivelano overfitting (train↓, val↑), underfitting (entrambi piatti) o un buon fitting (entrambi↓ bassi).

**Come interpretare**:
| Gap train–val | Tendenza | Lettura |
|---|---|---|
| Aperto (val > 1.2× train) | train↓, val↑ | Overfitting → early stopping ferma presto, ma i pesi migliori non sempre generalizzano |
| Chiuso, loss bassa | entrambi↓ | Good fitting |
| Chiuso, loss alta | entrambi piatti | Underfitting o rumore irriducibile |

> Nota: vale per il seed del checkpoint. La sezione B è **in sospeso** se `history` non è nel checkpoint (FASE 0 lo segnala).

In [ ]:
"""Section B — Training curves. In SOSPESO se history non è nel checkpoint."""
set_seed(SEED)

# --- Import needed functions ---
import csv
import matplotlib.pyplot as plt

history = ckpt.get("history")

if history is None:
    print("❌ 'history' non trovata nel checkpoint. Sezione B IN SOSPESO.")
    print("   → Modifica proposta: fit_autoencoder in autoencoder.py:803-812 salva history_seed{seed}.json")
    B_AVAILABLE = False
else:
    train_loss_hist = history.get("train_loss", [])
    val_loss_hist = history.get("val_loss", [])
    best_epoch = history.get("best_epoch", "N/A")
    B_AVAILABLE = len(train_loss_hist) > 0 and len(val_loss_hist) > 0

    if not B_AVAILABLE:
        print("❌ history presente ma vuoto. Sezione B IN SOSPESO.")
    else:
        print(f"✅ history trovata: {len(train_loss_hist)} epoche, best_epoch={best_epoch}")
        # Check coerenza: best_epoch <= len(history)
        if best_epoch != "N/A" and isinstance(best_epoch, int) and best_epoch > len(train_loss_hist):
            print(f"⚠️  best_epoch ({best_epoch}) > len(history) ({len(train_loss_hist)}) → BUG!")

        epochs = np.arange(1, len(train_loss_hist) + 1)

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        # Scala lineare
        ax = axes[0]
        ax.plot(epochs, train_loss_hist, label="Train Loss", color="#2196F3")
        ax.plot(epochs, val_loss_hist, label="Val Loss", color="#FF5722")
        if best_epoch != "N/A" and isinstance(best_epoch, int):
            ax.axvline(best_epoch, color="green", linestyle="--", label=f"best_epoch={best_epoch}")
        ax.set_xlabel("Epoca")
        ax.set_ylabel("Loss (MAE)")
        ax.set_title("Training curves — scala lineare")
        ax.legend()

        # Scala log
        ax = axes[1]
        ax.plot(epochs, train_loss_hist, label="Train Loss", color="#2196F3")
        ax.plot(epochs, val_loss_hist, label="Val Loss", color="#FF5722")
        if best_epoch != "N/A" and isinstance(best_epoch, int):
            ax.axvline(best_epoch, color="green", linestyle="--", label=f"best_epoch={best_epoch}")
        ax.set_xlabel("Epoca")
        ax.set_ylabel("Loss (MAE) — scala log")
        ax.set_title("Training curves — scala log")
        ax.set_yscale("log")
        ax.legend()

        plt.tight_layout()
        plt.savefig(DIAG_DIR / "ae_section_b_training_curves.png", dpi=120)
        plt.show()

        # Gap analysis
        final_train = train_loss_hist[-1]
        final_val = val_loss_hist[-1]
        best_val = min(val_loss_hist)
        min_train = min(train_loss_hist)
        print(f"  Train loss: inizio={train_loss_hist[0]:.6f}, fine={final_train:.6f}, min={min_train:.6f}")
        print(f"  Val loss:   inizio={val_loss_hist[0]:.6f}, fine={final_val:.6f}, min={best_val:.6f}")
        print(f"  Gap finale: {final_val - final_train:.6f}")

        if final_val > final_train * 1.2:
            print("\n  → Gap aperto: POSSIBILE OVERFITTING")
        elif final_val <= final_train * 1.2 and final_train < 0.1:
            print("\n  → Gap chiuso, loss bassa: buon fitting")
        else:
            print("\n  → Gap chiuso, loss alta: POSSIBILE UNDERFITTING")

        # Save CSV
        csv_path = DIAG_DIR / "ae_section_b_training_history.csv"
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["epoch", "train_loss", "val_loss"])
            for i, (tl, vl) in enumerate(zip(train_loss_hist, val_loss_hist), 1):
                writer.writerow([i, f"{tl:.6f}", f"{vl:.6f}"])
            writer.writerow([])
            writer.writerow(["best_epoch", best_epoch])
        print(f"\n  📊 CSV: {csv_path}")
        print("✅ Fine Sezione B")

if not B_AVAILABLE:
    print("\n⏸ Sezione B IN SOSPESO — salva history nel checkpoint per abilitarla.")

### Cosa osservo / Cosa concludo / Cosa NON posso concludere — Sezione B

**Cosa osservo**: le curve `train_loss` e `val_loss` per epoca (scala lineare e log), con `best_epoch` marcato; il gap tra train e val e la loro pendenza.

**Cosa concludo**: un gap aperto (val > 1.2× train) indica overfitting: l'early stopping salva i pesi migliori, che però potrebbero non generalizzare se il val_normal è driftato. Un gap chiuso con loss bassa indica underfitting o rumore irriducibile; con loss alta indica capacità insufficiente. Sezione B è **in sospeso** quando `history` è assente: in tal caso si affronta l'overfit test (F).

**Cosa NON posso concludere**: validità a lungo termine e su altri seed. Non so se `best_epoch` del checkpoint corrisponde al `best_epoch` del CSV di validazione di `run_experiment_ae.py`.

## C) Errore per canale — `compute_reconstruction_error(reduction='none')`

**Cosa consiste**: si calcola l'errore medio per ogni canale (sensore) su val_normal e val_anomaly, e se ne misURA l'AUC discriminante. Si aggrega l'AUC totale e si confronta AUC rimuovendo Acc/Gyro e action.

**Perché**: se l'AE impara (ratio < 0.3, sezione A) ma AUC ≈ 0.5, l'ampiezza non è per canale discriminativa. Serve capire **dove** l'informazione si perde: pochi canali discriminanti o tutti i canali a AUC ≈ 0.5.

**Come interpretare** (AUC per canale):
| AUC | Lettura |
|---|---||
| > 0.55 | canale discriminante (contribuisce) |
| < 0.45 | canale invertito (informazione presente, segno opposto) |
| 0.45–0.55 | canale non discriminante |

> `reduction='none'` (autoencoder.py:424) restituisce `diff` con la stessa shape di `x`; media su `W` → errore medio per canale per finestra.

In [ ]:
"""Section C — Errore per canale.

Usa compute_reconstruction_error(reduction="none") → (B, W, input_dim).
Media su W → (B, input_dim) = errore medio per canale per finestra.
Aggrega per val_normal e val_anomaly, calcola AUC per canale.

Salva: ae_section_c_channel_analysis.csv, ae_section_c_auc_per_channel.png,
       ae_section_c_channel_bar.png
"""
set_seed(SEED)

# --- Import needed functions ---
from src.utils.metrics import calculate_auc
import csv
import matplotlib.pyplot as plt

# --- Funzione helper: per-channel error da un loader ---
def compute_per_channel_error(loader, model, device, metric="mae"):
    """Restituisce (B, input_dim) = errore medio per canale per finestra."""
    model.eval()
    all_channel_errors = []
    with torch.no_grad():
        for batch in loader:
            x = batch[0] if isinstance(batch, (list, tuple)) else batch
            x = x.to(device)
            diff = model.compute_reconstruction_error(x, reduction="none", metric=metric)
            # diff shape: (B, W, input_dim) o (B, input_dim, W) — media su W
            diff_np = diff.cpu().numpy()
            # Media su W (axis=1 se layout è (B, W, D), axis=2 se (B, D, W))
            if diff_np.shape[1] == W:
                # (B, W, input_dim) → media su W
                channel_err = diff_np.mean(axis=1)  # (B, input_dim)
            else:
                # (B, input_dim, W) → media su W
                channel_err = diff_np.mean(axis=2)  # (B, input_dim)
            all_channel_errors.append(channel_err)
    return np.concatenate(all_channel_errors)

# --- Calcola errore per canale ---
print("Calcolo errore per canale (val_normal, val_anomaly, train)...")
channel_err_val_normal = compute_per_channel_error(val_normal_loader, model, DEVICE)
channel_err_val_anomaly = compute_per_channel_error(val_anomaly_loader, model, DEVICE)
channel_err_train = compute_per_channel_error(train_loader, model, DEVICE)

print(f"  channel_err_val_normal:  {channel_err_val_normal.shape}  (atteso: (B, 82))")
print(f"  channel_err_val_anomaly: {channel_err_val_anomaly.shape}")
print(f"  channel_err_train:       {channel_err_train.shape}")

# Media per canale (media su finestre)
err_normal_mean = channel_err_val_normal.mean(axis=0)   # (82,)
err_anomaly_mean = channel_err_val_anomaly.mean(axis=0) # (82,)
err_train_mean = channel_err_train.mean(axis=0)         # (82,)

# --- Tabella per canale ---
channel_names = [str(c) for c in selected_columns] if len(selected_columns) >= input_dim else [f"ch_{i}" for i in range(input_dim)]

# AUC per canale su validation
val_labels_2d = np.concatenate([
    np.zeros(channel_err_val_normal.shape[0]),
    np.ones(channel_err_val_anomaly.shape[0]),
])
auc_per_channel = []
for ch in range(input_dim):
    err_ch = np.concatenate([channel_err_val_normal[:, ch], channel_err_val_anomaly[:, ch]])
    auc = calculate_auc(val_labels_2d, err_ch)
    auc_per_channel.append(auc)

# --- Build channel CSV ---
csv_path = DIAG_DIR / "ae_section_c_channel_analysis.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["channel", "name", "err_train_mean", "err_val_normal", "err_val_anomaly",
                     "ratio_A_N", "%total_val", "roc_auc", "pr_auc"])
    total_val_err = (err_normal_mean + err_anomaly_mean).sum()
    for ch in range(input_dim):
        ratio_an = err_anomaly_mean[ch] / err_normal_mean[ch] if err_normal_mean[ch] > 0 else float("inf")
        pct = (err_normal_mean[ch] + err_anomaly_mean[ch]) / total_val_err * 100 if total_val_err > 0 else 0
        auc = auc_per_channel[ch]
        writer.writerow([ch, channel_names[ch], f"{err_train_mean[ch]:.6f}",
                        f"{err_normal_mean[ch]:.6f}", f"{err_anomaly_mean[ch]:.6f}",
                        f"{ratio_an:.4f}", f"{pct:.2f}",
                        f"{auc['roc_auc']:.4f}", f"{auc['pr_auc']:.4f}"])

print(f"📊 CSV salvato: {csv_path}")

# --- Plot 1: AUC per canale ---
fig, ax = plt.subplots(figsize=(14, 4))
aucs = [a["roc_auc"] for a in auc_per_channel]
colors = ["#2196F3" if 0.45 <= a <= 0.55 else ("#FF5722" if a < 0.5 else "#4CAF50") for a in aucs]
ax.bar(range(input_dim), aucs, color=colors, edgecolor="none")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.axhline(0.45, color="gray", linestyle=":", linewidth=0.5)
ax.axhline(0.55, color="gray", linestyle=":", linewidth=0.5)
ax.set_xlabel("Canale")
ax.set_ylabel("ROC-AUC")
ax.set_title(f"AUC per canale (W={W}) — rosso=discrimina (AUC>0.55), verde=invertita, grigio≈0.5")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(DIAG_DIR / "ae_section_c_auc_per_channel.png", dpi=120)
plt.show()

# --- Plot 2: Errore medio per canale (normal vs anomaly) ---
fig, ax = plt.subplots(figsize=(14, 4))
x_pos = np.arange(input_dim)
w_bar = 0.35
ax.bar(x_pos - w_bar/2, err_normal_mean, w_bar, label="Val normal", color="#2196F3", alpha=0.7)
ax.bar(x_pos + w_bar/2, err_anomaly_mean, w_bar, label="Val anomaly", color="#FF5722", alpha=0.7)
ax.set_xlabel("Canale")
ax.set_ylabel("MAE per canale (media finestre × timestep)")
ax.set_title(f"Errore medio per canale — val_normal vs val_anomaly (W={W})")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.savefig(DIAG_DIR / "ae_section_c_channel_bar.png", dpi=120)
plt.show()

# --- Summary: canali che separano ---
n_separating = sum(1 for a in aucs if a > 0.55)
n_inverted = sum(1 for a in aucs if a < 0.45)
n_useless = sum(1 for a in aucs if 0.45 <= a <= 0.55)
print(f"\n--- Riepilogo canali ---")
print(f"  Separano (AUC > 0.55):       {n_separating} canali")
print(f"  Invertiti (AUC < 0.45):      {n_inverted} canali")
print(f"  Inutili (0.45 ≤ AUC ≤ 0.55): {n_useless} canali")

# --- AUC totale "senza Acc/Gyro" e "senza action" ---
# Raggruppa canali per famiglia: cerca nomi
def find_channel_indices(name_patterns, channel_names):
    """Trova indici di canali il cui nome contiene una delle patterns."""
    idx = []
    for i, name in enumerate(channel_names):
        for pat in name_patterns:
            if pat in name:
                idx.append(i)
                break
    return idx

acc_idx = find_channel_indices(["Acc"], channel_names)
gyro_idx = find_channel_indices(["Gyro"], channel_names)
power_idx = find_channel_indices(["potenza", "power", "Power"], channel_names)
temp_idx = find_channel_indices(["temp", "Temp"], channel_names)
q_idx = find_channel_indices(["q1", "q2", "q3", "q4"], channel_names)
action_idx = find_channel_indices(["action", "Action"], channel_names)

all_val_err = np.concatenate([channel_err_val_normal, channel_err_val_anomaly])

def auc_without(indices_to_remove):
    keep = [i for i in range(input_dim) if i not in indices_to_remove]
    err_sum = all_val_err[:, keep].mean(axis=1)
    return calculate_auc(val_labels_2d, err_sum)

auc_full = calculate_auc(val_labels_2d, all_val_err.mean(axis=1))
auc_no_acc_gyro = auc_without(set(acc_idx + gyro_idx))
auc_no_action = auc_without(set(action_idx))

print(f"\n--- AUC totale (media canali) ---")
print(f"  Tutti i canali:              ROC-AUC={auc_full['roc_auc']:.4f}, PR-AUC={auc_full['pr_auc']:.4f}")
print(f"  Senza Acc+Gyro ({len(acc_idx)+len(gyro_idx)} rimossi): ROC-AUC={auc_no_acc_gyro['roc_auc']:.4f}, PR-AUC={auc_no_acc_gyro['pr_auc']:.4f}")
print(f"  Senza action ({len(action_idx)} rimossi): ROC-AUC={auc_no_action['roc_auc']:.4f}, PR-AUC={auc_no_action['pr_auc']:.4f}")

# --- Quote di errore totale per famiglia ---
families = [
    ("Acc", acc_idx), ("Gyro", gyro_idx), ("Power", power_idx),
    ("Temp", temp_idx), ("q1-q4", q_idx), ("action", action_idx),
]
print(f"\n--- Quote errore totale per famiglia (su val_anomaly) ---")
for name, idx in families:
    if len(idx) > 0:
        pct = err_anomaly_mean[idx].sum() / err_anomaly_mean.sum() * 100
        print(f"  {name:<8}: {len(idx):>3} canali, {pct:6.2f}% dell'errore totale")

print("\n✅ Fine Sezione C")

### Cosa osservo / Cosa concludo / Cosa NON posso concludere — Sezione C

**Cosa osservo**: per ogni canale l'errore medio su val_normal vs val_anomaly, il rapporto A/N, l'AUC di discriminazione e la percentuale sull'errore totale. L'AUC aggregata rimuovendo Acc/Gyro e action indica se quei canali spingono l'AUC nella direzione sbagliata.

**Cosa concludo**: pochi canali con AUC > 0.55 significa informazione concentrata in pochi sensori. Canali con AUC ≈ 0 (perfettamente invertiti) segnalano possibile problema di allineamento temporale o polarità, o rumore hardware. Se Acc/Gyro hanno alta percentuale di errore ma AUC ≈ 0.5, la rete li ricostruisce senza usarli discriminativamente. La media canale-hub può battere l'AUC full se la discriminatività è per canale, non aggregata.

**Cosa NON posso concludere**: se i canali invertiti sono rumore hardware o bug di allineamento — serve debug. Non generalizzabile a canali non nel checkpoint né a W diversi.

## D) Distribuzioni e drift temporale

**Cosa consiste**: (1) istogrammi dell'errore per finestra per split; (2) andamento del MAE nel tempo con media mobile; (3) cross-plot FPR(val_normal) vs FPR(test_normal) a soglie variabili.

**Perché**: gli istogrammi dicono se normal e anomalo hanno distribuzioni di errore sovrapposte. Il cross-plot FPR rileva **drift temporale**: se la stessa soglia dà FPR diversi su val e test, le distribuzioni hanno cambiato e l'AUC non generalizza.

**Come interpretare**:
| Evidenza | Lettura |
|---|---||
| Istogrammi sovrapposti, mediane simili | ampiezza non separa |
| Mediana val_anomaly ≫ val_normal | ampiezza anomala più alta |
| Punti cross-plot sulla bisettrice y=x | nessun drift → AUC stabile |
| Punti cross-plot deviati da y=x | drift temporale → AUC degrada su test |
| \|ΔAUC val–test\| > 0.1 | drift significativo |

In [ ]:
"""Section D — Distribuzioni e drift temporale.

1. Istogrammi errore per finestra: val_normal vs val_anomaly
2. Errore per finestra vs indice temporale (media mobile) per train, val_normal,
   test_normal, val_anomaly, test_anomaly
3. Cross-plot FPR(val_normal) vs FPR(test_normal) per vari threshold — drift test

Salva: ae_section_d_histograms.png, ae_section_d_temporal.png,
       ae_section_d_drift_crossplot.png, ae_section_d_drift_summary.csv
"""
set_seed(SEED)

# --- Import needed functions ---
from src.models.autoencoder import compute_errors_on_loader
from src.utils.metrics import calculate_auc
import csv
import matplotlib.pyplot as plt

# --- Compute per-window MAE errors for ALL splits ---
print("Calcolo errori per finestra per tutti gli split...")
err_train = compute_errors_on_loader(model, train_loader, DEVICE, metric="mae")
err_val_normal = compute_errors_on_loader(model, val_normal_loader, DEVICE, metric="mae")
err_val_anomaly = compute_errors_on_loader(model, val_anomaly_loader, DEVICE, metric="mae")
err_test_normal = compute_errors_on_loader(model, test_normal_loader, DEVICE, metric="mae")
err_test_anomaly = compute_errors_on_loader(model, test_anomaly_loader, DEVICE, metric="mae")

print(f"  train:        {len(err_train)}")
print(f"  val_normal:   {len(err_val_normal)}")
print(f"  val_anomaly:  {len(err_val_anomaly)}")
print(f"  test_normal:  {len(err_test_normal)}")
print(f"  test_anomaly: {len(err_test_anomaly)}")

# --- Test prevalence ---
test_prevalence = len(err_test_anomaly) / (len(err_test_normal) + len(err_test_anomaly))
print(f"\nPrevalenza test: {test_prevalence:.4f} ({len(err_test_anomaly)}/{len(err_test_normal)+len(err_test_anomaly)})")

# --- Plot 1: Istogrammi val_normal vs val_anomaly ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.hist(err_val_normal, bins=80, alpha=0.6, color="#2196F3", label="val_normal", density=True)
ax.hist(err_val_anomaly, bins=80, alpha=0.6, color="#FF5722", label="val_anomaly", density=True)
ax.axvline(np.median(err_val_normal), color="#1565C0", linestyle="--", linewidth=1.5, label=f"med norm={np.median(err_val_normal):.4f}")
ax.axvline(np.median(err_val_anomaly), color="#C62828", linestyle="--", linewidth=1.5, label=f"med anom={np.median(err_val_anomaly):.4f}")
ax.set_xlabel("MAE per finestra")
ax.set_ylabel("Densità")
ax.set_title("Distribuzione errore — val_normal vs val_anomaly")
ax.legend(fontsize=8)

ax = axes[1]
ax.hist(err_train, bins=80, alpha=0.5, color="#4CAF50", label="train", density=True)
ax.hist(err_val_normal, bins=80, alpha=0.5, color="#2196F3", label="val_normal", density=True)
ax.hist(err_test_normal, bins=80, alpha=0.5, color="#9C27B0", label="test_normal", density=True)
ax.set_xlabel("MAE per finestra")
ax.set_ylabel("Densità")
ax.set_title("Distribuzione errore — normal splits (train/val/test)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(DIAG_DIR / "ae_section_d_histograms.png", dpi=120)
plt.show()

# --- Plot 2: Errore vs tempo (indice finestra) — media mobile ---
fig, ax = plt.subplots(figsize=(16, 5))
window_ma = 200  # media mobile finestre

def rolling_mean(arr, w):
    """Rolling mean with window w. Expanding mean for first w-1 positions."""
    if len(arr) < w:
        return arr
    csum = np.cumsum(np.asarray(arr, dtype=float))
    result = np.empty(len(arr))
    # Expanding mean for first w-1 positions
    result[:w-1] = csum[:w-1] / np.arange(1, w)
    # Rolling mean for positions w-1 onwards (shapes: n-w+1 on both sides)
    result[w-1:] = (csum[w-1:] - np.concatenate([[0], csum[:-w]])) / w
    return result

splits_temporal = [("train", err_train), ("val_normal", err_val_normal),
                   ("test_normal", err_test_normal),
                   ("val_anomaly", err_val_anomaly), ("test_anomaly", err_test_anomaly)]

colors = {"train": "#4CAF50", "val_normal": "#2196F3", "test_normal": "#9C27B0",
          "val_anomaly": "#FF5722", "test_anomaly": "#E65100"}

# Offset per non sovrapporre (usa indice cumulativo)
cursor = 0
x_labels = []
x_positions = []
for name, errs in splits_temporal:
    rm = rolling_mean(errs, min(window_ma, len(errs)))
    x = np.arange(cursor, cursor + len(rm)) + 1
    ax.plot(x, rm, label=name, color=colors[name], linewidth=1)
    ax.fill_between(x, errs, alpha=0.05, color=colors[name])
    cursor += len(errs)
    x_labels.append(name)
    x_positions.append(cursor - len(errs) // 2)

ax.set_xlabel("Indice finestra (sequenziale)")
ax.set_ylabel("MAE (media mobile)")
ax.set_title(f"Errore per finestra vs tempo (media mobile w={window_ma}) — W={W}")
ax.set_yscale("log")
ax.set_xticks(x_positions)
ax.set_xticklabels(x_labels, rotation=45)
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.savefig(DIAG_DIR / "ae_section_d_temporal.png", dpi=120)
plt.show()

# --- Plot 3: Cross-plot FPR(val_normal) vs FPR(test_normal) — drift test ---
# Per ogni threshold (percentile di val_normal), calcola FPR su val_normal e su test_normal
thresholds = np.percentile(err_val_normal, np.arange(1, 100))
fpr_val = []
fpr_test = []
for t in thresholds:
    # FPR = falsi positivi / normali totali
    fp_val = np.sum(err_val_normal >= t) / len(err_val_normal)
    fp_test = np.sum(err_test_normal >= t) / len(err_test_normal)
    fpr_val.append(fp_val)
    fpr_test.append(fp_test)

fpr_val = np.array(fpr_val)
fpr_test = np.array(fpr_test)

# Mediana e deviazione
median_diff = np.median(np.abs(fpr_val - fpr_test))
max_diff = np.max(np.abs(fpr_val - fpr_test))

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_val, fpr_test, "o-", markersize=2, linewidth=0.8, alpha=0.6, label="soglie per soglia")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="y=x (nessun drift)")
ax.fill_between([0, 1], [0, 1], [0.1, 1.1], alpha=0.15, color="orange", label="zona drift >10%")
ax.set_xlabel("FPR(val_normal)")
ax.set_ylabel("FPR(test_normal)")
ax.set_title(f"Drift test: FPR(val_normal) vs FPR(test_normal)\n"
             f"mediana|ΔFPR|={median_diff:.4f}, max|ΔFPR|={max_diff:.4f}")
ax.legend(fontsize=8)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)
plt.tight_layout()
plt.savefig(DIAG_DIR / "ae_section_d_drift_crossplot.png", dpi=120)
plt.show()

# --- CSV drift summary ---
csv_path = DIAG_DIR / "ae_section_d_drift_summary.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["split", "n_windows", "mean_MAE", "median_MAE", "std_MAE", "p05", "p95"])
    for name, errs in splits_temporal:
        writer.writerow([name, len(errs), f"{errs.mean():.6f}", f"{np.median(errs):.6f}",
                        f"{errs.std():.6f}", f"{np.percentile(errs, 5):.6f}", f"{np.percentile(errs, 95):.6f}"])
    writer.writerow([])
    writer.writerow(["metric", "value"])
    writer.writerow(["median_abs_fpr_diff", f"{median_diff:.6f}"])
    writer.writerow(["max_abs_fpr_diff", f"{max_diff:.6f}"])
    writer.writerow(["threshold_count", len(thresholds)])

print(f"\n📊 CSV drift: {csv_path}")
print(f"  Mediana |FPR_val - FPR_test|: {median_diff:.6f}")
print(f"  Max |FPR_val - FPR_test|:     {max_diff:.6f}")
if max_diff < 0.15:
    print("  → Drift sostenuto < 15% su FPR: distribuzioni simili")
elif max_diff < 0.30:
    print("  → Drift moderato (15-30%): val/test leggermente diversi")
else:
    print("  → Drift forte (> 30%): val/test distribuzioni molto diverse → AUC degrada")

# --- AUC comparison val vs test ---
test_labels = np.concatenate([np.zeros(len(err_test_normal)), np.ones(len(err_test_anomaly))])
test_errors = np.concatenate([err_test_normal, err_test_anomaly])
test_auc = calculate_auc(test_labels, test_errors)

val_labels_all = np.concatenate([np.zeros(len(err_val_normal)), np.ones(len(err_val_anomaly))])
val_errors_all = np.concatenate([err_val_normal, err_val_anomaly])
val_auc = calculate_auc(val_labels_all, val_errors_all)

print(f"\n--- AUC val vs test (MAE per finestra) ---")
print(f"  Val:      ROC-AUC={val_auc['roc_auc']:.4f}, PR-AUC={val_auc['pr_auc']:.4f} (prev={val_prevalence:.4f})")
print(f"  Test:     ROC-AUC={test_auc['roc_auc']:.4f}, PR-AUC={test_auc['pr_auc']:.4f} (prev={test_prevalence:.4f})")
if abs(val_auc['roc_auc'] - test_auc['roc_auc']) > 0.1:
    print("  → Delta AUC > 0.1: possibile drift temporale")
else:
    print("  → Delta AUC < 0.1: distribuzioni coerenti tra val e test")

print("\n✅ Fine Sezione D")

### Cosa osservo / Cosa concludo / Cosa NON posso concludere — Sezione D

**Cosa osservo**: istogrammi dell'errore per finestra (val_normal vs val_anomaly, e normal splits tra loro), andamento temporale con media mobile, e cross-plot FPR(val_normal) vs FPR(test_normal). Il confronto AUC val vs test mostra se il degrado è consistente.

**Cosa concludo**: istogrammi sovrapposti con mediane simili confermano che l'ampiezza non separa le classi. Il cross-plot deviato dalla bisettrice y=x (max|ΔFPR| alto) indica drift: se il drift riguarda i normal split, è un cambiamento nella distribuzione del modello o del dato. Un |ΔAUC| val–test > 0.1 conferma che l'AUC non generalizza. L'andamento temporale della media MAE (crescente) suggerisce drift graduale.

**Cosa NON posso concludere**: causa del drift (condizioni operative, degrado sensore) senza metadati temporali. Drift su altri seed non controllato.

## E) Baseline senza rete — Statistiche su raw features

**Cosa consiste**: si calcolano statistiche banali per finestra (deviazione standard media, media |z-score|, valore assoluto massimo) e se ne confronta l'AUC con quella dell'AE.

**Perché**: se una statistica grezza batte l'AE, l'informazione discriminante **esiste** nel segnale ma l'AE non la estrae → spreca informazione. Se invece anche le baseline hanno AUC ≈ 0.5, nessuna informazione a questa finestratura.

**Statistiche** (z-score da train, applicato a tutti gli split per evitare leakage):
| Statistica | Che cattura |
|---|---|
| std_media | ampiezza variazione per finestra |
| mean_abs_z | intensità \|z\|-score medio per finestra |
| max_abs_z | spike / valore assoluto massimo per finestra |

> Se `max_abs_z` batte l'AE, l'informazione è nel segnale ma la loss MAE della rete non la usa.

In [ ]:
"""Section E — Baseline senza rete.

Statistiche banali (std, mean|z|, max|z|) su raw features, z-score da train.
Confronta AUC di queste statistiche con AUC dell'AE (sezione A).

Salva: ae_section_e_baseline_results.csv
"""
set_seed(SEED)

# --- Import needed functions ---
from src.utils.metrics import calculate_auc
import csv
import matplotlib.pyplot as plt

# --- Compute z-score statistics per window ---
# Usa build_loader con lo stesso W del checkpoint
def compute_zscore_stats(loader, channel_mean, channel_std):
    """Per finestra: std_media, mean_abs_z, max_abs_z (media su canali × timestep)."""
    std_list, mean_abs_z_list, max_abs_z_list = [], [], []
    for batch in loader:
        x = batch[0] if isinstance(batch, (list, tuple)) else batch
        x_np = x.numpy()  # (B, W, input_dim)
        # z-score: solo per canali con std > 0
        z = (x_np - channel_mean) / np.where(channel_std > 1e-8, channel_std, 1.0)
        abs_z = np.abs(z)

        # Media su finestra, canali, timestep → un valore per finestra
        std_per_window = x_np.std(axis=(1, 2))       # (B,)
        mean_abs_z_per_window = abs_z.mean(axis=(1, 2))  # (B,)
        max_abs_z_per_window = abs_z.max(axis=(1, 2))    # (B,)

        std_list.append(std_per_window)
        mean_abs_z_list.append(mean_abs_z_per_window)
        max_abs_z_list.append(max_abs_z_per_window)

    return (np.concatenate(std_list), np.concatenate(mean_abs_z_list),
            np.concatenate(max_abs_z_list))

# --- Compute channel stats from train (media, std per canale) ---
train_flat_all = train_data  # (n_samples, n_features)
channel_mean = train_flat_all.mean(axis=0)  # (n_features,)
channel_std = train_flat_all.std(axis=0)    # (n_features,)
print(f"z-score stats calcolate su train: mean range [{channel_mean.min():.4f}, {channel_mean.max():.4f}], "
      f"std range [{channel_std.min():.4f}, {channel_std.max():.4f}]")

# --- Compute stats per split ---
print("\nCalcolo statistiche baseline (std, |z|, max|z|) per split...")
std_train, mz_train, mxz_train = compute_zscore_stats(train_loader, channel_mean, channel_std)
std_val_n, mz_val_n, mxz_val_n = compute_zscore_stats(val_normal_loader, channel_mean, channel_std)
std_val_a, mz_val_a, mxz_val_a = compute_zscore_stats(val_anomaly_loader, channel_mean, channel_std)
std_test_n, mz_test_n, mxz_test_n = compute_zscore_stats(test_normal_loader, channel_mean, channel_std)
std_test_a, mz_test_a, mxz_test_a = compute_zscore_stats(test_anomaly_loader, channel_mean, channel_std)

# --- AUC comparison on validation ---
val_labels = np.concatenate([np.zeros(len(std_val_n)), np.ones(len(std_val_a))])

stats = {
    "std_mean":   np.concatenate([std_val_n, std_val_a]),
    "mean_abs_z": np.concatenate([mz_val_n, mz_val_a]),
    "max_abs_z":  np.concatenate([mxz_val_n, mxz_val_a]),
}

# AUC dell'AE (già calcolato in sezione A)
# Usiamo err_val_normal e err_val_anomaly calcolati in sezione D
# Se sezione D non è stata eseguita, ricalcoliamo qui
try:
    err_val_normal
    err_val_anomaly
except NameError:
    from src.models.autoencoder import compute_errors_on_loader
    err_val_normal = compute_errors_on_loader(model, val_normal_loader, DEVICE, metric="mae")
    err_val_anomaly = compute_errors_on_loader(model, val_anomaly_loader, DEVICE, metric="mae")

auc_ae_val = calculate_auc(val_labels, np.concatenate([err_val_normal, err_val_anomaly]))

csv_path = DIAG_DIR / "ae_section_e_baseline_results.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["stat_name", "roc_auc_val", "pr_auc_val", "prevalence_val"])
    writer.writerow(["AE_reconstruction_error", f"{auc_ae_val['roc_auc']:.4f}", f"{auc_ae_val['pr_auc']:.4f}", f"{val_prevalence:.4f}"])
    for stat_name, stat_scores in stats.items():
        auc = calculate_auc(val_labels, stat_scores)
        writer.writerow([stat_name, f"{auc['roc_auc']:.4f}", f"{auc['pr_auc']:.4f}", f"{val_prevalence:.4f}"])

print(f"\n📊 CSV: {csv_path}")
print(f"\n--- AUC baseline vs rete (validation) ---")
print(f"  AE reconstruction:  ROC-AUC={auc_ae_val['roc_auc']:.4f}, PR-AUC={auc_ae_val['pr_auc']:.4f}  (prevalenza={val_prevalence:.4f})")

best_baseline_auc = 0
best_baseline_name = ""
for stat_name, stat_scores in stats.items():
    auc = calculate_auc(val_labels, stat_scores)
    marker = " ← BATTE L'AE!" if auc['roc_auc'] > auc_ae_val['roc_auc'] else ""
    print(f"  {stat_name:<14}: ROC-AUC={auc['roc_auc']:.4f}, PR-AUC={auc['pr_auc']:.4f}{marker}")
    if auc['roc_auc'] > best_baseline_auc:
        best_baseline_auc = auc['roc_auc']
        best_baseline_name = stat_name

if best_baseline_auc > auc_ae_val['roc_auc']:
    print(f"\n  → {best_baseline_name} batte l'AE: l'AE SPRECA informazione disponibile nel segnale!")
else:
    print(f"\n  → L'AE batte o equala le baseline: l'informazione grezza non è sufficiente")

# --- Plot comparison ---
fig, ax = plt.subplots(figsize=(10, 4))
names = ["AE_recon"] + list(stats.keys())
aucs = [auc_ae_val['roc_auc']] + [calculate_auc(val_labels, s)['roc_auc'] for s in stats.values()]
colors = ["#2196F3"] + ["#FF9800", "#FF9800", "#FF9800"]
bars = ax.bar(range(len(names)), aucs, color=colors)
ax.axhline(0.5, color="gray", linestyle="--", label="random (0.5)")
ax.axhline(val_prevalence, color="red", linestyle=":", label=f"prevalenza ({val_prevalence:.3f})")
for bar, v in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f"{v:.3f}", ha="center", fontsize=9)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=20)
ax.set_ylabel("ROC-AUC (validation)")
ax.set_title("Confronto: baseline statistiche vs AE")
ax.legend(fontsize=8)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(DIAG_DIR / "ae_section_e_baseline_comparison.png", dpi=120)
plt.show()

print("\n✅ Fine Sezione E")

### Cosa osservo / Cosa concludo / Cosa NON posso concludere — Sezione E

**Cosa osservo**: confronto AUC (validation) tra AE e baseline statistiche (std, media |z|, max |z|), con prevalenza mostrata. Se una baseline batte l'AE, l'informazione è nel segnale ma non viene estratta.

**Cosa concludo**: se una baseline grezza (es. max |z|) batte l'AE, l'informazione discriminante è **presente** nel segnale ma la rete non la utilizza: il colpevole è la fine loss (MAE sensibile ai canali rumorosi) o il dato (canali rumorosi dominano). Se invece l'AE batte le baseline, l'informazione non è a questa finestratura e serve cercare in drift (D), overfit (F) o segno invertito (C).

**Cosa NON posso concludere**: perché l'AE non usa il segnale (verifica F: se l'architettura converge, è il dato, non la rete). Le baseline valgono per W del checkpoint.

## F) Overfit test — capacità del modello vs rumore irriducibile

**Cosa consiste**: si addestrano 3 modelli piccoli (512 finestre contigue, 300 epoche, MAE loss) con configurazioni di capacità crescente, per verificare se riescono a memorizzare i dati (MAE ≈ 0).

**Perché**: se anche la configurazione a massima capacità non memorizza, non è un collo di bottiglia o bug architetturale — è **rumore irriducibile** nel dato, **drift**, o **fine loss**. Se invece memorizza, il modello è corretto e il problema è altrove.

**Come interpretare**:
| Config (i) W16/lat256 (massima capacità) | Config (iii) W128/lat16 | Diagnosi |
|---|---|---|
| memorizza | memorizza | nessun collo di bottiglia → problema in dato/drift/loss |
| memorizza | NO | collo di bottiglia: latent=16 insufficiente per W=128 |
| NO memorizza | — | bug architetturale grave |

> *Eccezione* all'obbligo read-only: addestramento inside-notebook; salva solo JSON/PNG in `reports/diagnostics/`.

In [ ]:
"""Section F — Overfit test.

3 config addestrati inside-notebook su 512 finestre contigue (shuffle=False).
weight_decay=0, Adam lr=1e-3, 300 ep, batch 128, MAE loss, set_seed(42).
Salva: overfit_config_i.json, overfit_config_ii.json, overfit_config_iii.json
       overfit_per_channel_mae.npy
       overfit_summary.csv
"""
import json as _json

set_seed(SEED)

# --- Import needed functions ---
from src.models.autoencoder import SequenceAutoencoder, fit_autoencoder
from src.data.dataset import KukaDataset
import csv
import matplotlib.pyplot as plt

# --- Prepare 512 contiguous windows per config ---
def get_512_windows(data, window_size):
    """Restituisce 512 finestre consecutive shape (512, W, input_dim)."""
    ds = KukaDataset(data, window_size=window_size, stride=1, label=0, drop_incomplete=True)
    windows = np.stack([ds[i][0] for i in range(min(512, len(ds)))])
    return windows

train_windows_16 = get_512_windows(train_data, window_size=16)
train_windows_128 = get_512_windows(train_data, window_size=128)

print(f"Finestre estratte: W16={train_windows_16.shape}, W128={train_windows_128.shape}")

# --- MAE_zero per ogni config ---
chan_mean_all = channel_mean  # (82,)
mae_zero_16 = np.abs(train_windows_16 - chan_mean_all).mean(axis=(1, 2)).mean()
mae_zero_128 = np.abs(train_windows_128 - chan_mean_all).mean(axis=(1, 2)).mean()

print(f"\nMAE_zero (baseline teorica): W16={mae_zero_16:.6f}, W128={mae_zero_128:.6f}")

# --- Overfit configs ---
configs = [
    {"name": "config_i",   "W": 16,  "latent": 256, "data": train_windows_16},
    {"name": "config_ii",  "W": 16,  "latent": 16,  "data": train_windows_16},
    {"name": "config_iii", "W": 128, "latent": 16,  "data": train_windows_128},
]

OVERFIT_EPOCHS = 300
OVERFIT_BS = 128

results_f = {}
for cfg in configs:
    print(f"\n{'='*50}")
    print(f"Training: {cfg['name']} — W={cfg['W']}, latent={cfg['latent']}")
    print(f"{'='*50}")

    set_seed(42)

    x_tensor = torch.from_numpy(cfg["data"]).float()
    ds = torch.utils.data.TensorDataset(x_tensor)
    loader = torch.utils.data.DataLoader(ds, batch_size=OVERFIT_BS, shuffle=False)

    model_of = SequenceAutoencoder(input_dim=82, window_size=cfg["W"], latent_dim=cfg["latent"])
    print(f"  Parametri modello: {sum(p.numel() for p in model_of.parameters())}")

    history_of = fit_autoencoder(
        model=model_of,
        train_loader=loader,
        val_loader=loader,
        epochs=OVERFIT_EPOCHS,
        learning_rate=1e-3,
        weight_decay=0.0,
        patience=999,
        min_delta=0.0,
        device=DEVICE,
        loss="mae",
    )

    final_loss = history_of["train_loss"][-1]
    best_loss = min(history_of["train_loss"])
    mz = mae_zero_16 if cfg["W"] == 16 else mae_zero_128
    print(f"  Final train loss: {final_loss:.6f}")
    print(f"  Best train loss:  {best_loss:.6f}")
    print(f"  MAE_zero:         {mz:.6f}")
    print(f"  ratio AE/MAE_zero: {final_loss / mz:.4f}")

    if final_loss < 0.05:
        print(f"  ✅ OVERFITTATO (MAE < 0.05)")
    else:
        print(f"  ❌ NON overfittato (MAE >= 0.05)")

    json_path = DIAG_DIR / f"overfit_{cfg['name']}.json"
    with open(json_path, "w") as f:
        _json.dump({
            "config": cfg["name"], "W": cfg["W"], "latent": cfg["latent"],
            "epochs": OVERFIT_EPOCHS, "batch_size": OVERFIT_BS,
            "lr": 1e-3, "weight_decay": 0.0, "seed": 42,
            "train_loss": history_of["train_loss"],
            "val_loss": history_of["val_loss"],
            "best_epoch": history_of["best_epoch"],
            "final_train_loss": final_loss,
            "best_train_loss": best_loss,
            "mae_zero": float(mz),
        }, f, indent=2)
    print(f"  📊 Salvato: {json_path}")

    results_f[cfg["name"]] = {
        "W": cfg["W"], "latent": cfg["latent"],
        "final_train_loss": final_loss,
        "best_train_loss": best_loss,
        "overfitted": final_loss < 0.05,
        "history": history_of,
        "model_state": {k: v.clone() for k, v in model_of.state_dict().items()},
    }

# --- Per-channel MAE dopo overfit (config_i, W=16) ---
print(f"\n{'='*50}")
print("Per-channel MAE after overfit (config_i, W=16)")
print(f"{'='*50}")

set_seed(42)
model_f_of = SequenceAutoencoder(input_dim=82, window_size=16, latent_dim=256)
model_f_of.load_state_dict(results_f["config_i"]["model_state"])
model_f_of.to(DEVICE)
model_f_of.eval()

with torch.no_grad():
    x = torch.from_numpy(train_windows_16).float().to(DEVICE)
    diff = model_f_of.compute_reconstruction_error(x, reduction="none", metric="mae")
    per_channel_mae = diff.cpu().numpy().mean(axis=(0, 1))  # (82,)

np.save(DIAG_DIR / "overfit_per_channel_mae.npy", per_channel_mae)
print(f"  Salvato: {DIAG_DIR / 'overfit_per_channel_mae.npy'}")

high_noise = np.where(per_channel_mae > 0.3)[0]
print(f"  Canali con MAE > 0.3 (rumore irriducibile): {len(high_noise)}/{input_dim}")

# CSV per-channel MAE
csv_path = DIAG_DIR / "ae_section_f_channel_mae.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["channel", "name", "per_channel_mae"])
    for ch in range(input_dim):
        cname = channel_names[ch] if ch < len(channel_names) else f"ch_{ch}"
        writer.writerow([ch, cname, f"{per_channel_mae[ch]:.6f}"])
print(f"  CSV: {csv_path}")

# --- Plot: training curves ---
fig, ax = plt.subplots(figsize=(10, 5))
for cfg in configs:
    tl = results_f[cfg["name"]]["history"]["train_loss"]
    ax.plot(range(1, len(tl)+1), tl, label=f"{cfg['name']} (W={cfg['W']},lat={cfg['latent']})")
ax.axhline(0.05, color="red", linestyle="--", label="soglia MAE 0.05")
ax.set_xlabel("Epoca")
ax.set_ylabel("Train Loss (MAE)")
ax.set_title("Overfit test — training curves (3 config)")
ax.set_yscale("log")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(DIAG_DIR / "ae_section_f_overfit_curves.png", dpi=120)
plt.show()

# --- Summary diagnosis ---
print(f"\n{'='*50}")
print("DIAGNOSI OVERFIT")
print(f"{'='*50}")
i_ok = results_f["config_i"]["overfitted"]
ii_ok = results_f["config_ii"]["overfitted"]
iii_ok = results_f["config_iii"]["overfitted"]
print(f"  (i)   W=16,  lat=256:  overfit={i_ok} (MAE={results_f['config_i']['final_train_loss']:.6f})")
print(f"  (ii)  W=16,  lat=16:   overfit={ii_ok} (MAE={results_f['config_ii']['final_train_loss']:.6f})")
print(f"  (iii) W=128, lat=16:   overfit={iii_ok} (MAE={results_f['config_iii']['final_train_loss']:.6f})")

if not i_ok:
    print("\n  → BUG ARCHITETTURALE GRAVE: nemmeno (i) non overfitta!")
elif i_ok and ii_ok and iii_ok:
    print("\n  → NESSUN collo di bottiglia: capacità sufficiente. Problema altrove (drift, loss, dato).")
elif i_ok and ii_ok and not iii_ok:
    print("\n  → COLLO DI BOTTIGLIA: latent=16 OK per W=16, insufficiente per W=128.")
    print("     Suggerimento: aumentare latent_dim in params_validated_ae.yaml.")
else:
    print("\n  → Caso imprevisto: verifica manuale.")

print(f"\n  Canali rumore irriducibile (MAE > 0.3): {len(high_noise)}/{input_dim}")

# Summary CSV
with open(DIAG_DIR / "ae_section_f_overfit_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["config", "W", "latent", "final_mae", "overfitted", "mae_zero", "ratio"])
    for cfg in configs:
        r = results_f[cfg["name"]]
        mz = mae_zero_16 if cfg["W"] == 16 else mae_zero_128
        writer.writerow([cfg["name"], cfg["W"], cfg["latent"],
                        f"{r['final_train_loss']:.6f}", r["overfitted"],
                        f"{mz:.6f}", f"{r['final_train_loss']/mz:.4f}"])
    writer.writerow(["high_noise_channels_count", "", "", "", "", len(high_noise), ""])
    writer.writerow(["total_channels", "", "", "", "", "", input_dim])

print(f"\n  📊 Summary: {DIAG_DIR / 'ae_section_f_overfit_summary.csv'}")
print("\n✅ Fine Sezione F")

### Cosa osservo / Cosa concludo / Cosa NON posso concludere — Sezione F

**Cosa osservo**: le curve di training loss delle 3 configurazioni e i final MAE rispetto alla soglia di sovraffiggimento (MAE < 0.05); i canali con MAE > 0.3 dopo l'overfit (rumore irriducibile).

**Cosa concludo**: se la configurazione a massima capacità (i) overfitta (MAE < 0.05) ma (iii) non overfitta, c'è un **collo di bottiglia** (`latent_dim` insufficiente per `W=128`). Se invece **nessuna** configurazione overfitta (né (i) con capacità massima), il modello converge ma si piatta: la causa è **rumore irriducibile** su canali specifici (MAE > 0.3 anche overfittando) che domina la loss. Se tutte overfittano, l'architettura è corretta e il problema è drift/loss (vedi D/E).

**Cosa NON posso concludere**: effetto a lungo termine (512 finestre, 1 seed, solo train). Il rumore su N canali non implica rumore su tutti i canali né che rimuoverli migliori la discriminatività globale.

## G) Sintesi finale — Tabella ipotesi / N_eff / Appendice

**Cosa consiste**: (1) raccoglie le evidenze da A–F in una **tabella ipotesi → verdetto**; (2) stima `N_eff` con due approcci; (3) appendice operativa con soglia calibrata, matrice di confusione, baseline `tutto anomalo`.

**Perché**: unire le evidenze rende comparabili le ipotesi e isola la causa radice. `N_eff` dice quanti campioni indipendenti il modello vede davvero (a causa dell'overlap delle finestre). L'appendice confronta il modello con la soglia ottimale contro la baseline banale `tutto anomalo` (F1 = prevalenza), come minimo accettabile.

**6 ipotesi da valutare**:
| # | Ipotesi | Evidenza | Verdetto |
|---|---|---|---|
| 1 | Rete non impara | MAE_AE/MAE_zero | A |
| 2 | Collo di bottiglia latent=16/W=128 | overfit config (iii) vs (i) | F |
| 3 | Rumore irriducibile | canali MAE > 0.3 | C / F |
| 4 | Pochi indipendenti (N_eff basso) | N_eff | G (sottostante) |
| 5 | Drift temporale | max\|ΔFPR\| val–test | D |
| 6 | Segno invertito / canali AUC ≈ 0.5 | AUC per canale | C / E |

> Soglia di calibrazione su validation, mai su test. Test serve solo a conferma. La causa radice emerge dalla combinazione, non da un singolo test.

In [ ]:
"""Section G — Sintesi finale.

1. Tabella ipotesi → verdetto (usa risultati di A–F)
2. N_eff: due approcci (overlap teorico + ρ₁ empirica)
3. Appendice: soglia calibrata, matrice confusione, baseline "tutto anomalo"

Salva: ae_section_g_synthesis.csv, ae_section_g_neff.csv,
       ae_section_g_confusion_matrix.csv, ae_section_g_threshold.png
"""
set_seed(SEED)

# --- Import needed functions ---
from src.utils.metrics import calculate_auc, calculate_metrics, find_optimal_threshold
from sklearn.metrics import confusion_matrix
import csv
import matplotlib.pyplot as plt

# ============================================================
# 1. Raccolta evidenze da A–F (riutilizza variabili già calcolate)
# ============================================================
print("=" * 60)
print("1. TABELLA IPOTESI → VERDETTO")
print("=" * 60)

# --- Ipotesi 1: rete non impara ---
# ratio MAE_AE/MAE_zero calcolato in sezione A
ratio_val_anom = mae_ae_val_anomaly.mean() / mae_zero_val_anomaly.mean() if 'mae_ae_val_anomaly' in dir() else 0
h1_threshold_ratio = 0.8
h1_verdict = "non impara" if ratio_val_anom > h1_threshold_ratio else "impara qualcosa"
print(f"\n  Ipotesi 1: rete non impara (ratio MAE_AE/MAE_zero > {h1_threshold_ratio})")
print(f"    ratio val_anomaly = {ratio_val_anom:.4f}")
print(f"    → {h1_verdict}")

# --- Ipotesi 2: collo di bottiglia latent=16 per W=128 ---
# Da sezione F: config_iii overfittato?
if 'results_f' in dir():
    if results_f["config_i"]["overfitted"] is False and results_f["config_ii"]["overfitted"] is False and results_f["config_iii"]["overfitted"] is False:
        # Nessuno overfitta neanche con capacity massima (config_i: W=16, latent=256)
        # -> NON è collo di bottiglia: è rumore irriducibile o canali non modellabili.
        h2_verdict = "rumore irriducibile / canali non modellabili (nessun modello overfitta)"
    elif results_f["config_i"]["overfitted"] is True and results_f["config_ii"]["overfitted"] is True and not results_f["config_iii"]["overfitted"]:
        # (i)+(ii) overfittano, (iii) no -> collo di bottiglia latent=16 per W=128
        h2_verdict = "collo di bottiglia (latent=16 insufficiente per W=128)"
    elif results_f["config_i"]["overfitted"] is True and results_f["config_ii"]["overfitted"] is True and results_f["config_iii"]["overfitted"] is True:
        h2_verdict = "OK (nessun collo di bottiglia; problema in drift/rumore/dato)"
    else:
        h2_verdict = "inconcludibile (caso imprevisto)"
else:
    h2_verdict = "IN SOSPESO (sezione F non eseguita)"
print(f"\n  Ipotesi 2: collo di bottiglia (latent=16 insufficiente per W=128)")
print(f"    config_iii overfittato={results_f['config_iii']['overfitted'] if 'results_f' in dir() else '?'}")
print(f"    → {h2_verdict}")

# --- Ipotesi 3: rumore irriducibile Acc/Gyro ---
# Da sezione F: canali con MAE > 0.3, e da sezione C: AUC Acc/Gyro
if 'high_noise' in dir():
    n_acc_gyro_noise = len([ch for ch in high_noise if any(pat in channel_names[ch] for pat in ["Acc", "Gyro"])])
    h3_verdict = "rumore irriducibile" if n_acc_gyro_noise > 0 else "non confermato dai canali"
else:
    h3_verdict = "IN SOSPESO"
    n_acc_gyro_noise = "?"
print(f"\n  Ipotesi 3: rumore irriducibile Acc/Gyro")
print(f"    canali Acc/Gyro con MAE > 0.3 (dopo overfit): {n_acc_gyro_noise}")
print(f"    → {h3_verdict}")

# --- Ipotesi 4: pochi campioni indipendenti (N_eff) ---
n_train_windows = len(train_loader.dataset)
n_val_windows = len(val_normal_loader.dataset) + len(val_anomaly_loader.dataset)

# (a) Overlap teorico: N_eff ≈ N_windows / (W/2)
W_for_neff = W
n_eff_overlap_train = n_train_windows / (W_for_neff / 2)
n_eff_overlap_val = n_val_windows / (W_for_neff / 2)

# (b) ρ₁ empirica: lag-1 autocorrelation sul vettore errore per finestra
# Usa err_train (già calcolato in sezione D)
if 'err_train' in dir() and len(err_train) > 2:
    # Lag-1 autocorrelation
    e = err_train
    e_mean = e.mean()
    e_var = e.var()
    if e_var > 1e-12:
        rho1 = np.corrcoef(e[:-1], e[1:])[0, 1]
    else:
        rho1 = 0.0
    n_eff_rho_train = n_train_windows * (1 - rho1) / (1 + rho1)
else:
    rho1 = 0.5  # fallback
    n_eff_rho_train = n_train_windows * (1 - rho1) / (1 + rho1)

n_eff_rho_val = n_val_windows * (1 - rho1) / (1 + rho1)

print(f"\n  Ipotesi 4: pochi campioni indipendenti (N_eff basso)")
print(f"    n_train_windows = {n_train_windows}, W = {W_for_neff}")
print(f"    (a) Overlap:     N_eff ≈ N/(W/2) = {n_eff_overlap_train:.0f} (train), {n_eff_overlap_val:.0f} (val)")
print(f"    (b) ρ₁ empirica: ρ₁ = {rho1:.4f}")
print(f"                  N_eff = N·(1-ρ)/(1+ρ) = {n_eff_rho_train:.0f} (train), {n_eff_rho_val:.0f} (val)")
n_eff_min = min(n_eff_overlap_train, n_eff_rho_train)
h4_verdict = "N_eff basso (≤ 1000)" if n_eff_min < 1000 else "N_eff sufficiente"
print(f"    → {h4_verdict}")

# --- Ipotesi 5: drift temporale ---
# Da sezione D: max_diff FPR
if 'max_diff' in dir():
    h5_verdict = "drift" if max_diff > 0.15 else "nessun drift"
else:
    h5_verdict = "IN SOSPESO"
print(f"\n  Ipotesi 5: drift temporale (val vs test)")
print(f"    max|ΔFPR| = {max_diff:.6f}" if 'max_diff' in dir() else "    (vedi sezione D)")
print(f"    → {h5_verdict}")

# --- Ipotesi 6: lentezza non aumenta errore / canali AUC ≈ 0.5 ---
# Da sezione C: n_useless canali
if 'n_useless' in dir():
    h6_verdict = "confermato" if n_useless > input_dim * 0.9 else "non confermato"
    print(f"\n  Ipotesi 6: lentezza non aumenta errore (canali AUC ≈ 0.5)")
    print(f"    canali inutili (0.45 ≤ AUC ≤ 0.55): {n_useless}/{input_dim}")
    print(f"    → {h6_verdict}")
else:
    h6_verdict = "IN SOSPESO"
    n_useless = "?"

# --- Save synthesis CSV ---
synth_path = DIAG_DIR / "ae_section_g_synthesis.csv"
with open(synth_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["ipotesi", "descrizione", "evidenza", "verdetto"])
    writer.writerow([1, "Rete non impara", f"MAE_AE/MAE_zero(val_anom)={ratio_val_anom:.4f}", h1_verdict])
    writer.writerow([2, "Collo di bottiglia latent=16/W=128",
                     f"config_iii overfit={results_f['config_iii']['overfitted'] if 'results_f' in dir() else '?'}", h2_verdict])
    writer.writerow([3, "Rumore irriducibile Acc/Gyro", f"{n_acc_gyro_noise} canali > 0.3", h3_verdict])
    writer.writerow([4, "N_eff basso",
                     f"overlap={n_eff_overlap_train:.0f}, rho1={rho1:.4f}, N_eff_rho={n_eff_rho_train:.0f}", h4_verdict])
    writer.writerow([5, "Drift temporale", f"max|ΔFPR|={max_diff:.4f}" if 'max_diff' in dir() else "vedi D", h5_verdict])
    writer.writerow([6, "Lentezza non aumenta errore", f"{n_useless} canali AUC≈0.5", h6_verdict])
print(f"\n  📊 Synthesis: {synth_path}")

# --- Save N_eff CSV ---
neff_path = DIAG_DIR / "ae_section_g_neff.csv"
with open(neff_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["approach", "n_windows", "n_eff", "rho1"])
    writer.writerow(["overlap_theoretical", n_train_windows, f"{n_eff_overlap_train:.0f}", "N/A"])
    writer.writerow(["rho1_empirical", n_train_windows, f"{n_eff_rho_train:.0f}", f"{rho1:.4f}"])
    writer.writerow(["overlap_theoretical_val", n_val_windows, f"{n_eff_overlap_val:.0f}", "N/A"])
    writer.writerow(["rho1_empirical_val", n_val_windows, f"{n_eff_rho_val:.0f}", f"{rho1:.4f}"])
print(f"  📊 N_eff: {neff_path}")

# ============================================================
# 2. Appendice: soglia calibrata + matrice confusione + baseline "tutto anomalo"
# ============================================================
print(f"\n{'='*60}")
print("2. APPENDICE — Soglia, matrice confusione, baseline 'tutto anomalo'")
print(f"{'='*60}")

# Soglia calibrata su validation (val_normal + val_anomaly)
val_errors_combined = np.concatenate([err_val_normal, err_val_anomaly])
val_labels_combined = np.concatenate([
    np.zeros(len(err_val_normal), dtype=np.int32),
    np.ones(len(err_val_anomaly), dtype=np.int32),
])

best_thresh, best_metrics = find_optimal_threshold(val_labels_combined, val_errors_combined)
print(f"  Soglia ottimale (F1): {best_thresh:.6f}")
print(f"  Metriche a soglia ottimale: {best_metrics}")

# Matrice confusione a soglia ottimale (su validation)
val_preds = (val_errors_combined >= best_thresh).astype(np.int32)
cm_val = confusion_matrix(val_labels_combined, val_preds)
print(f"\n  Matrice confusione (validation, soglia={best_thresh:.6f}):")
print(f"    [[TN={cm_val[0,0]}, FP={cm_val[0,1]}]")
print(f"    [ FN={cm_val[1,0]}, TP={cm_val[1,1]}]]")

# Matrice confusione su TEST (stessa soglia)
test_errors_combined = np.concatenate([err_test_normal, err_test_anomaly])
test_labels_combined = np.concatenate([
    np.zeros(len(err_test_normal), dtype=np.int32),
    np.ones(len(err_test_anomaly), dtype=np.int32),
])

test_preds = (test_errors_combined >= best_thresh).astype(np.int32)
cm_test = confusion_matrix(test_labels_combined, test_preds)
test_metrics = calculate_metrics(test_labels_combined, test_preds)
test_auc = calculate_auc(test_labels_combined, test_errors_combined)

print(f"\n  Matrice confusione (TEST, stessa soglia={best_thresh:.6f}):")
print(f"    [[TN={cm_test[0,0]}, FP={cm_test[0,1]}]")
print(f"    [ FN={cm_test[1,0]}, TP={cm_test[1,1]}]]")
print(f"  Test metrics: {test_metrics}")
print(f"  Test AUC: ROC={test_auc['roc_auc']:.4f}, PR={test_auc['pr_auc']:.4f}")

# Baseline "tutto anomalo"
n_test = len(test_labels_combined)
n_anom_test = test_labels_combined.sum()
acc_all_anom = (n_anom_test) / n_test
prec_all_anom = n_anom_test / n_test  # tutti predetti come anomali
rec_all_anom = 1.0
f1_all_anom = 2 * prec_all_anom * rec_all_anom / (prec_all_anom + rec_all_anom) if prec_all_anom > 0 else 0

print(f"\n  Baseline 'tutto anomalo' (su test):")
print(f"    accuracy={acc_all_anom:.4f}, precision={prec_all_anom:.4f}, recall={rec_all_anom:.4f}, f1={f1_all_anom:.4f}")
print(f"    (prevalenza test = {test_prevalence:.4f})")
print(f"    → F1 = prev = {f1_all_anom:.4f} (conferma/ipotesi F1≈0.64)")

# Salva matrice confusione CSV
cm_path = DIAG_DIR / "ae_section_g_confusion_matrix.csv"
with open(cm_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["set", "threshold", "TN", "FP", "FN", "TP", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"])
    # Validation
    writer.writerow(["val", f"{best_thresh:.6f}", cm_val[0,0], cm_val[0,1], cm_val[1,0], cm_val[1,1],
                    f"{best_metrics['accuracy']:.4f}", f"{best_metrics['precision']:.4f}",
                    f"{best_metrics['recall']:.4f}", f"{best_metrics['f1']:.4f}",
                    f"{auc_ae['roc_auc']:.4f}" if 'auc_ae' in dir() else "?",
                    f"{auc_ae['pr_auc']:.4f}" if 'auc_ae' in dir() else "?"])
    # Test
    writer.writerow(["test", f"{best_thresh:.6f}", cm_test[0,0], cm_test[0,1], cm_test[1,0], cm_test[1,1],
                    f"{test_metrics['accuracy']:.4f}", f"{test_metrics['precision']:.4f}",
                    f"{test_metrics['recall']:.4f}", f"{test_metrics['f1']:.4f}",
                    f"{test_auc['roc_auc']:.4f}", f"{test_auc['pr_auc']:.4f}"])
    # Baseline tutto anomalo
    writer.writerow(["all_anom_test", "N/A", 0, 0, 0, n_anom_test,
                    f"{acc_all_anom:.4f}", f"{prec_all_anom:.4f}", f"{rec_all_anom:.4f}",
                    f"{f1_all_anom:.4f}", "0.5", "0.5"])
print(f"\n  📊 Confusion matrix: {cm_path}")

# Plot: FPR threshold curve + confusion matrix
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.hist(err_val_normal, bins=80, alpha=0.5, color="#2196F3", density=True, label="val_normal")
ax.hist(err_val_anomaly, bins=80, alpha=0.5, color="#FF5722", density=True, label="val_anomaly")
ax.axvline(best_thresh, color="green", linestyle="--", linewidth=1.5, label=f"soglia F1={best_thresh:.4f}")
ax.set_xlabel("MAE per finestra")
ax.set_ylabel("Densità")
ax.set_title(f"Soglia calibrata (val) — prevalenza={val_prevalence:.4f}")
ax.legend(fontsize=8)

ax = axes[1]
im = ax.imshow(cm_val, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Normale", "Anomalo"])
ax.set_yticklabels(["Normale", "Anomalo"])
ax.set_title(f"Matrice confusione (val)\nF1={best_metrics['f1']:.3f}")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_val[i, j]), ha="center", va="center", fontsize=12, fontweight="bold")

ax = axes[2]
im = ax.imshow(cm_test, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Normale", "Anomalo"])
ax.set_yticklabels(["Normale", "Anomalo"])
ax.set_title(f"Matrice confusione (test)\nF1={test_metrics['f1']:.3f}, AUC={test_auc['roc_auc']:.3f}")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_test[i, j]), ha="center", va="center", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig(DIAG_DIR / "ae_section_g_threshold_cm.png", dpi=120)
plt.show()

print(f"\n📊 Plot: {DIAG_DIR / 'ae_section_g_threshold_cm.png'}")
print("\n" + "=" * 60)
print("✅ FINE DIAGNOSI — tutti i risultati in:", DIAG_DIR)
print("=" * 60)

### Cosa osservo / Cosa concludo / Cosa NON posso concludere — Sezione G

**Cosa osservo**: la tabella ipotesi→verdetto raccoglie le evidenze da A–F. `N_eff` è stimata con due approcci (overlap teorico `N/(W/2)` e ρ₁ empirica). L'appendice mostra soglia calibrata, matrice di confusione su val e test, e la baseline `tutto anomalo` (F1 = prevalenza) come minimo accettabile.

**Cosa concludo**: le ipotesi confermate indicano la sezione di riferimento dove indagare la causa. La combinazione delle evidenze (non un singolo test) punta alla causa radice. La soglia calibrata su validation non va mai ricalibrata su test: il test serve solo a conferma. `N_eff` basso indebolisce la statistica delle soglie.

**Cosa NON posso concludere**: che la causa sia unica e sufficiente — le ipotesi sono spesso interagenti (rumore + drift + N_eff basso). Non valuto configurazioni architetturali alternative né riaddestramenti futuri: questo notebook diagnostica i pesi esistenti, non ne addestra di nuovi.

## Conclusione generale — Diagnostica modello AE

Questo notebook offre un flusso di lavoro **read-only e riproducibile** (seed = 42) per diagnosticare un autoencoder su dati temporali multivariati. Sette test (FASE 0 + A–F) indagano ciascuno una singola ipotesi di guasto; la sezione G li riunisce in una tabella ipotesi→verdetto e chiude con un'appendice operativa.

**Causa radice — schema generale**. Qualsiasi under-performance si riduce a una di queste cause, ciascuna mappata su una sezione:
| Causa | Che cosa indaga | Sezione | Rimedio tipico |
|---|---|---|---|
| L'AE non impara nulla | MAE reale ≈ MAE del modello stupido | A | verificare architettura / loss |
| Collo di bottiglia architetturale | l'AE sovraffiglia? (config iii vs i) | F | aumentare `latent_dim` / capacità |
| Rumore irriducibile | canali MAE > 0.3 dopo overfit | C / F | filtrare canali rumorosi, loss robusta |
| Informazione sprecata | baseline grezza batte l'AE | E | revisione loss, feature engineering, o sostituzione AE |
| Segno invertito / canali non discriminativi | AUC canale ≈ 0 o ≈ 0.5 | C | controllare allineamento temporale e polarità |
| Drift temporale | max\|ΔFPR\| val↔test | D | soglia drift-aware, stride maggiore, normalizzazione rolling |
| Pochi indipendenti (N_eff basso) | finestre altamente sovrapposte | G | stride = W//2, early stopping aggressivo |

**Come leggere la diagnosi**: ogni sezione (A–G) produce artefatti (CSV/PNG/JSON) in `reports/diagnostics/model_diagnosis/` e conclude con *Cosa osservo / Cosa concludo / Cosa NON posso concludere*. La sezione G raccoglie i verdetti in una tabella ipotesi→verdetto, stima `N_eff`, e fornisce l'appendice: soglia calibrata su validation, matrice di confusione (val + test) e baseline `tutto anomalo` (F1 = prevalenza) come riferimento minimo. **La causa radice emerge dalla combinazione delle evidenze, non da un singolo test.**

> Riportare sempre la **prevalenza** accanto alle AUC (necessaria per leggere la PR-AUC). Per robustezza, ripetere la diagnosi su più seed e mediare i verdetti.